In [14]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from utils.settings import *
from dotenv import load_dotenv
from langchain_mistralai import MistralAIEmbeddings
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from utils.preprocessing import TEXT_COLUMNS, build_event_texts, save_embeddings


In [2]:
# Chargement du model embedding
load_dotenv()

embedding_model = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=os.environ["MISTRAL_API"]
)

D:\Formation\OpenClassroom\Ai engineer\Projets\7 - RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Charger le dataset
df = pd.read_csv(f"{datas_folder}/events_clean.csv", sep=";",dtype={"location_postalcode": "string"})

# Afficher ses dimensions
print("Shape :", df.shape)

Shape : (11234, 56)


In [8]:
# Choix des colonnes utile pour l'embedding
remplissage = pd.DataFrame({
    "Valeurs présentes": df.notna().sum(),
    "Valeurs manquantes": df.isna().sum(),
    "Remplissage (%)": (df.notna().mean() * 100).round(2)
})

remplissage = remplissage.sort_values(
    by="Remplissage (%)",
    ascending=False
)

display(remplissage[:30])


,Valeurs présentes,Valeurs manquantes,Remplissage (%)
uid,11234,0,100.00
slug,11234,0,100.00
canonicalurl,11234,0,100.00
location_uid,11234,0,100.00
updatedat,11234,0,100.00
daterange_fr,11234,0,100.00
firstdate_begin,11234,0,100.00
firstdate_end,11234,0,100.00
location_name,11234,0,100.00
location_coordinates,11234,0,100.00


In [ ]:
%%sql


| Colonnes                                           | Justification                                                                                 |
|----------------------------------------------------|-----------------------------------------------------------------------------------------------|
| `title_fr`, `description_fr`, `longdescription_fr` | Informations principales des événements.                                                      |
| `keywords_fr`                                      | Colonne pensée exprès pour la description d'un événement.                                     |
| `daterange_fr`                                     | Permet d'identifier les dates et horaires des événements.                                     |
| `location_name`, `location_address`, 'location_district'              | Permet de rechercher des événements selon leur localisation géographique.                     |
| `conditions_fr`                                    | Fournit des informations sur les tarifs, les réservations et les conditions de participation. |
| `age_min`, `age_max`                               | Permet d'identifier les événements adaptés à certaines tranches d'âge.                        |
| `accessibility_label_fr`, `location_access_fr`     | Fournit des informations sur l'accessibilité des événements et les moyens d'accès.            |

In [9]:
cols = TEXT_COLUMNS
[col for col in cols if col not in df.columns]


[]

In [12]:
texts = build_event_texts(df, cols)
# Exemple de sortie
print(texts[0])


C:\Users\alexa_5h5xfmj\AppData\Local\Temp\ipykernel_36064\1066388880.py:8: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  f"{col}: {BeautifulSoup(value, 'html.parser').get_text(' ', strip=True)}"


title_fr: FIAP Jean Monnet
description_fr: 1 ERP de type O avec activité L, N, P et R, 1 ERP de type Let R, 1 ERP de type O
longdescription_fr: Le FIAP Paris est un Centre International de Sejour situé en plein cœur de Paris dans le 14ème arrondissement. Son objectif tel qu’il a été imaginé et conçu à la fin de la Seconde Guerre Mondiale est de favoriser la rencontre et les échanges entre les jeunes et de lutter contre toutes les formes de discriminations. Pour servir ce projet, le FIAP Paris exerce deux activités principales : - Il héberge et accueille en priorité des jeunes et des groupes de jeunes de toutes nationalités. - Il permet à des organismes de tous horizons d’organiser dans ses salles des séminaires, des congrès et des réunions. Le FIAP propose à ses hôtes un grand nombre d’activités culturelles et pédagogiques : expositions, concerts etc. Il organise également pour ses hôtes en longs séjours des moments privilégiés et particuliers qui sont autant d’occasions pour les jeune

In [15]:
output_dir = Path(datas_folder)
embeddings = save_embeddings(
    texts, embedding_model, output_dir / "events_embeddings.npy", batch_size=50
)


100%|██████████| 225/225 [10:39<00:00,  2.84s/it]


In [16]:
# Vérification des embeddings
embeddings = np.load(output_dir / "events_embeddings.npy")

print("Nombre d'événements :", len(df))
print("Nombre d'embeddings :", len(embeddings))
print("Dimensions :", embeddings.shape)

Nombre d'événements : 11234
Nombre d'embeddings : 11234
Dimensions : (11234, 1024)
